### Classes


<p style="font-size:50px">
<span style="font-size:200px">
&#9703;
</span>
Les classes
</p>

---



Sous-sections :
[Classes et objets](#classes)&nbsp;|
[Déplaçons nos points](#deplace)&nbsp;|
[Faisons des carrés !](#carres)&nbsp;
[Jouons avec une fractale](#fractale)&nbsp;|

<a id="classes"></a>
### Classes et objets

Voici une définition de la classe `Point` :

In [ ]:
import matplotlib.pyplot as plt

class Point(object):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def dessine_sur(self, ax):
        ax.plot([self.x], [self.y], 'o', lw=2)[0]

    def deplace(self, dx, dy):
        pass
        # <- completer ici
        self.x = self.x + dx
        self.y = self.y + dy

    def __repr__(self):
        return f'({self.x}, {self.y})'



En programmation objet, 
une classe comme `Point` joue le rôle à la fois de type de données et de module :
- en tant que __type__, elle définit un ensemble de valeurs (ses instances) et des opérations qu'on peut effectuer sur ces valeurs ;
- en tant que __module__, elle définit un espace de noms. On peut accéder aux noms de cet espace avec la notation pointée.

__La classe comme un module__

A l'aide de la fonction `vars()`, afficher l'espace de noms défini par la classe `Point` et vérifier qu'il contient bien l'opération `dessine_sur`.

In [ ]:
# <- completer ici

Supposons que je rajoute l'attribut `u` suivant dans l'espace de noms de `Point` :

In [ ]:
Point.u = 'une valeur'

Afficher à nouveau l'espace de noms de la classe `Point` et vérifier que le nom `u` est maintenant bien présent.

In [ ]:
# <- completer ici

__La classe comme type__

La classe définit un nouveau type de données :
- les *instances* de la classe sont les valeurs
- les méthodes définies dans la classe sont les *opérations* qu'on peut effectuer sur ces valeurs

Quel est le type de la classe `Point` ?

In [ ]:
# <- completer ici

A l'aide de la fonction `dir()`, afficher l'ensemble des noms définis dans la classe `Point`.

In [ ]:
# <- completer ici

Certains de ces noms (lesquels ?), appelés __attributs__, référencent d'autres objets.

Certains de ces noms (lesquels ?), appelés __méthodes__, référencent des fonctions qui permettent d'échanger des messages avec d'autres objets.  Ces méthodes correspondent aux *opérations* du type de données.


Quel est le type de l'objet `a` ci-dessous ?

In [ ]:
a = Point(-3,0)
# <- completer ici

En utilisant la fonction `vars()` (ou simplement `a.__dict__`), afficher la structure interne de l'objet `a`.


In [ ]:
# <- completer ici

A l'aide de la fonction `dir()`, afficher l'ensemble des noms **dir**ectement accessibles sur l'objet `a`, c'est-à-dire l'ensemble des attributs, méthodes définies dans la classe de l'objet et dans ses superclasses.

Vérifier qu'il contient bien en particulier la méthode `dessine_sur` et l'attribut `u` hérités de `Point`.

In [ ]:
# <- completer ici

On accède aux attributs et méthodes d'une instance à l'aide de la notation pointée :

In [ ]:
a.x

In [ ]:
a = Point(-3,0)
fig, ax = plt.subplots(figsize=(8,2),dpi=72)
a.dessine_sur(ax)

<a id="deplace"></a>
### Déplaçons nos points

Pour afficher nos figures, on crée une classe `Dessin` :

In [ ]:
class Dessin(object):

    def __init__(self):
        fig = plt.figure(figsize=(10, 10),dpi=72)
        ax = fig.add_subplot(xlim=(-10,10),ylim=(-10,10))
        ax.set_aspect('equal')
        ax.set_axis_off()
        self.fig = fig
        self.ax = ax

    def dessine(self, p):
        p.dessine_sur(self.ax)

Ajouter une méthode d'instance `deplace` à la classe `Point` qui prend deux arguments `dx` et `dy` et qui déplace le point de `dx` en abscisse et de `dy` en ordonnée.

Après la modification, le code suivant doit afficher sur le dessin 18 points également espacés :

In [ ]:
dessin = Dessin()
a = Point(-9,0)
for i in range(18):
    a.deplace(1,0)
    dessin.dessine(a)

### Faisons des carrés !

On décide qu'un polygone est une liste de points :

In [ ]:
class Polygone(list[Point]):

    def dessine_sur(self, ax):
        (x,y) = ([e.x for e in self], [e.y for e in self])  
        ax.plot(x, y, '-', lw=2)[0]

Je peux maintenant afficher un polygone sur mon dessin :

In [ ]:
a = Point(-3,0)
b = Point(-3,3)
c = Point(3,3)
p = Polygone([a,b,c,a])
dessin = Dessin()
dessin.dessine(p)

On voudrait pouvoir créer des carrés en spécifiant uniquement le centre `c` et la taille du côté `r`.

Compléter la classe `Carre` ci-dessous pour qu'on puisse créer un carré de cette façon.

In [ ]:
class Carre(Polygone):
    def __init__(self, centre, r):
        ...
        # <- completer ici

Le code ci-dessous doit afficher un carré de centre `(0,0)` et de côté 12 :

In [ ]:
carre = Carre((0,0),12)
Dessin().dessine(carre)

<a id="fractale"></a>
### Jouons avec une fractale

Voici un type de polygone particulier appelé `VonKoch` qui décompose chacune de ses arêtes `n_iter` fois à l'aide de la décomposition de von Koch :

In [ ]:
import numpy as np
class VonKoch(Polygone):
    def __init__(self, *args, n_iter=3):
        super().__init__(*args)
        for _ in range(n_iter):
            self.decompose()

    def decompose(self):
        n = self.__len__()
        i = n 
        while i>1:
            u,v = self[i-2:i]
            a = np.array([u.x,u.y])
            b = np.array([v.x,v.y])
            d = np.linalg.norm(a-b)/3
            new_u = a + (b-a)/3
            new_v = a + 2*(b-a)/3
            theta = np.pi/3
            rot = np.array([[np.cos(theta), -np.sin(theta)],[np.sin(theta), np.cos(theta)]])
            new_w = new_u + rot @ (new_v-new_u)
            self.insert(i-1,Point(*list(new_v)))
            self.insert(i-1,Point(*list(new_w)))
            self.insert(i-1,Point(*list(new_u)))
            i = i - 1

On peut créer ce type de polygone en lui passant une liste de points et un niveau de décomposition `n_iter` de la manière suivante :

In [ ]:
a = Point(-8,0)
b = Point(8,0)
p = VonKoch([a,b], n_iter=4)
Dessin().dessine(p)

Dessiner ce polygone en prenant différentes valeurs de `n_iter` et différentes listes de points.

In [ ]:
# <- completer ici

Utiliser la classe `Carre` précédente pour dessiner un flocon de von Koch sur un carré de centre `(0,0)` et de côté 12.

In [ ]:
# <- completer ici